# 🏪 End-to-End Supply Chain & Profitability Analysis
### Global Superstore Dataset

**Author:** Supply Chain Analytics Team  
**Dataset:** Sample Superstore (9,994 records, 21 columns)  
**Objective:** Identify supply chain inefficiencies and profitability drivers

---

## 📋 Table of Contents
1. [Environment Setup & Data Loading](#1-environment-setup)
2. [Data Exploration (EDA)](#2-exploratory-data-analysis)
3. [Data Cleaning & Preprocessing](#3-data-cleaning)
4. [Feature Engineering](#4-feature-engineering)
5. [Key Performance Indicators (KPIs)](#5-kpis)
6. [Shipping Efficiency Analysis](#6-shipping-efficiency)
7. [Profitability Analysis](#7-profitability-analysis)
8. [SQL-Based Aggregations](#8-sql-aggregations)
9. [Visualizations & Dashboards](#9-visualizations)
10. [Insights & Recommendations](#10-recommendations)

## 1. Environment Setup & Data Loading

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
import sqlite3

warnings.filterwarnings('ignore')

# Plot aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})
COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#FF9800', '#9C27B0']

print("✅ Libraries loaded successfully")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────────────────────
# Update this path to your local file location
FILE_PATH = 'Sample_-_Superstore.csv'

df_raw = pd.read_csv(FILE_PATH, encoding='latin1')
print(f"✅ Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BASIC OVERVIEW
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Rows    : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]}")
print()
print("--- Column Data Types ---")
print(df_raw.dtypes)
print()
print("--- Missing Values ---")
print(df_raw.isnull().sum())

In [ ]:
# ─────────────────────────────────────────────────────────────
# DESCRIPTIVE STATISTICS
# ─────────────────────────────────────────────────────────────
df_raw[['Sales', 'Quantity', 'Discount', 'Profit']].describe().round(2)

In [ ]:
# Unique value counts for categorical columns
cat_cols = ['Region', 'Category', 'Sub-Category', 'Ship Mode', 'Segment']
for col in cat_cols:
    print(f"{col}: {df_raw[col].unique().tolist()}")

## 3. Data Cleaning & Preprocessing

In [ ]:
# ─────────────────────────────────────────────────────────────
# CLEANING PIPELINE
# ─────────────────────────────────────────────────────────────
df = df_raw.copy()

# 1. Drop duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"✅ Duplicates removed: {before - len(df)}")

# 2. Parse date columns
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', dayfirst=False)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='mixed', dayfirst=False)
print("✅ Date columns parsed to datetime")

# 3. Strip whitespace from string columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())
print("✅ Whitespace stripped from string columns")

# 4. Verify nulls after cleaning
print(f"\n✅ Total nulls after cleaning: {df.isnull().sum().sum()}")
print(f"✅ Final shape: {df.shape}")

## 4. Feature Engineering

In [ ]:
# ─────────────────────────────────────────────────────────────
# NEW FEATURES
# ─────────────────────────────────────────────────────────────

# Delivery time (days)
df['Delivery_Time'] = (df['Ship Date'] - df['Order Date']).dt.days

# Profit margin (%)
df['Profit_Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

# Revenue per unit
df['Revenue_Per_Unit'] = (df['Sales'] / df['Quantity']).round(2)

# Order year and month
df['Order_Year']  = df['Order Date'].dt.year
df['Order_Month'] = df['Order Date'].dt.month
df['Order_Quarter'] = df['Order Date'].dt.to_period('Q').astype(str)

# Discount bucket
df['Discount_Bucket'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0.0, 0.2, 0.4, 1.0],
    labels=['No Discount', 'Low (0–20%)', 'Mid (20–40%)', 'High (>40%)']
)

# Profitability flag
df['Is_Profitable'] = df['Profit'] > 0

print("✅ New features created:")
new_cols = ['Delivery_Time', 'Profit_Margin', 'Revenue_Per_Unit',
            'Order_Year', 'Order_Month', 'Order_Quarter',
            'Discount_Bucket', 'Is_Profitable']
print(df[new_cols].head())

## 5. Key Performance Indicators (KPIs)

In [ ]:
# ─────────────────────────────────────────────────────────────
# OVERALL KPIs
# ─────────────────────────────────────────────────────────────
kpis = {
    'Total Orders'          : df['Order ID'].nunique(),
    'Total Sales ($)'       : round(df['Sales'].sum(), 2),
    'Total Profit ($)'      : round(df['Profit'].sum(), 2),
    'Overall Profit Margin' : f"{df['Profit'].sum() / df['Sales'].sum() * 100:.2f}%",
    'Avg Delivery Time (d)' : round(df['Delivery_Time'].mean(), 1),
    'Total Customers'       : df['Customer ID'].nunique(),
    'Total Products'        : df['Product ID'].nunique(),
    'Loss-Making Orders'    : int((~df['Is_Profitable']).sum()),
}

print("=" * 45)
print("   📊  BUSINESS KPI SUMMARY")
print("=" * 45)
for k, v in kpis.items():
    print(f"  {k:<30} {v}")
print("=" * 45)

In [ ]:
# ─────────────────────────────────────────────────────────────
# YEARLY KPI TREND
# ─────────────────────────────────────────────────────────────
yearly = df.groupby('Order_Year').agg(
    Total_Sales   = ('Sales', 'sum'),
    Total_Profit  = ('Profit', 'sum'),
    Total_Orders  = ('Order ID', 'nunique'),
    Avg_Delivery  = ('Delivery_Time', 'mean')
).round(2).reset_index()
yearly['Profit_Margin_%'] = (yearly['Total_Profit'] / yearly['Total_Sales'] * 100).round(2)
print(yearly.to_string(index=False))

## 6. Shipping Efficiency Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────
# SHIPPING MODE ANALYSIS
# ─────────────────────────────────────────────────────────────
ship_stats = df.groupby('Ship Mode').agg(
    Order_Count   = ('Order ID', 'count'),
    Avg_Delivery  = ('Delivery_Time', 'mean'),
    Min_Delivery  = ('Delivery_Time', 'min'),
    Max_Delivery  = ('Delivery_Time', 'max'),
    Total_Sales   = ('Sales', 'sum'),
    Total_Profit  = ('Profit', 'sum')
).round(2).reset_index()
ship_stats['Profit_Margin_%'] = (ship_stats['Total_Profit'] / ship_stats['Total_Sales'] * 100).round(2)
ship_stats.sort_values('Avg_Delivery')

In [ ]:
# ─────────────────────────────────────────────────────────────
# SHIPPING VISUALIZATIONS
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Shipping Efficiency Analysis', fontsize=16, fontweight='bold', y=1.01)

# Chart 1 - Average delivery time by ship mode
ax1 = axes[0]
bars = ax1.barh(ship_stats['Ship Mode'], ship_stats['Avg_Delivery'],
                color=COLORS[:4], edgecolor='white', linewidth=0.8)
ax1.set_xlabel('Avg Delivery Days')
ax1.set_title('Average Delivery Time by Ship Mode')
for bar, val in zip(bars, ship_stats['Avg_Delivery']):
    ax1.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}d', va='center', fontsize=10)

# Chart 2 - Order distribution by ship mode
ax2 = axes[1]
ax2.pie(ship_stats['Order_Count'], labels=ship_stats['Ship Mode'],
        autopct='%1.1f%%', colors=COLORS[:4], startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax2.set_title('Order Distribution by Ship Mode')

plt.tight_layout()
plt.savefig('shipping_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Delivery time distribution per ship mode
fig, ax = plt.subplots(figsize=(12, 5))
for i, mode in enumerate(df['Ship Mode'].unique()):
    subset = df[df['Ship Mode'] == mode]['Delivery_Time']
    ax.hist(subset, bins=15, alpha=0.6, label=mode, color=COLORS[i], edgecolor='white')
ax.set_xlabel('Delivery Time (days)')
ax.set_ylabel('Frequency')
ax.set_title('Delivery Time Distribution by Ship Mode')
ax.legend()
plt.tight_layout()
plt.savefig('delivery_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Profitability Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────
# PROFITABILITY BY REGION
# ─────────────────────────────────────────────────────────────
region_stats = df.groupby('Region').agg(
    Sales  = ('Sales', 'sum'),
    Profit = ('Profit', 'sum')
).round(2).reset_index()
region_stats['Margin_%'] = (region_stats['Profit'] / region_stats['Sales'] * 100).round(2)
region_stats.sort_values('Profit', ascending=False)

In [ ]:
# ─────────────────────────────────────────────────────────────
# PROFITABILITY BY CATEGORY & SUB-CATEGORY
# ─────────────────────────────────────────────────────────────
cat_stats = df.groupby(['Category', 'Sub-Category']).agg(
    Sales  = ('Sales', 'sum'),
    Profit = ('Profit', 'sum'),
    Orders = ('Order ID', 'count')
).round(2).reset_index()
cat_stats['Margin_%'] = (cat_stats['Profit'] / cat_stats['Sales'] * 100).round(2)
cat_stats.sort_values('Profit').head(10)  # Bottom 10

In [ ]:
# ─────────────────────────────────────────────────────────────
# DISCOUNT vs PROFIT ANALYSIS
# ─────────────────────────────────────────────────────────────
disc_profit = df.groupby('Discount_Bucket').agg(
    Avg_Profit = ('Profit', 'mean'),
    Avg_Sales  = ('Sales', 'mean'),
    Count      = ('Order ID', 'count')
).round(2).reset_index()
print(disc_profit)

In [ ]:
# ─────────────────────────────────────────────────────────────
# PROFITABILITY VISUALIZATIONS
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Profitability Analysis Dashboard', fontsize=18, fontweight='bold')

# 1. Sales vs Profit by Region
ax1 = axes[0, 0]
x = range(len(region_stats))
w = 0.35
ax1.bar([i - w/2 for i in x], region_stats['Sales']/1000, w, label='Sales ($K)', color='#2196F3')
ax1.bar([i + w/2 for i in x], region_stats['Profit']/1000, w, label='Profit ($K)', color='#4CAF50')
ax1.set_xticks(x)
ax1.set_xticklabels(region_stats['Region'])
ax1.set_ylabel('Amount ($K)')
ax1.set_title('Sales vs Profit by Region')
ax1.legend()

# 2. Profit margin by category
ax2 = axes[0, 1]
cat_margin = df.groupby('Category')['Profit_Margin'].mean().reset_index()
colors_cm = ['#4CAF50' if x > 0 else '#FF5722' for x in cat_margin['Profit_Margin']]
ax2.bar(cat_margin['Category'], cat_margin['Profit_Margin'], color=colors_cm, edgecolor='white')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_ylabel('Avg Profit Margin (%)')
ax2.set_title('Average Profit Margin by Category')

# 3. Discount vs Profit scatter
ax3 = axes[1, 0]
sample = df.sample(n=min(2000, len(df)), random_state=42)
ax3.scatter(sample['Discount'], sample['Profit'], alpha=0.3, c='#9C27B0', s=10)
ax3.axhline(0, color='red', linewidth=1, linestyle='--', label='Break-even')
ax3.set_xlabel('Discount Rate')
ax3.set_ylabel('Profit ($)')
ax3.set_title('Discount vs Profit (sampled)')
ax3.legend()

# 4. Top 10 loss-making sub-categories
ax4 = axes[1, 1]
loss_sub = cat_stats.sort_values('Profit').head(8)
bar_colors = ['#FF5722' if v < 0 else '#4CAF50' for v in loss_sub['Profit']]
ax4.barh(loss_sub['Sub-Category'], loss_sub['Profit']/1000, color=bar_colors)
ax4.axvline(0, color='black', linewidth=0.8)
ax4.set_xlabel('Profit ($K)')
ax4.set_title('Lowest Profit Sub-Categories')

plt.tight_layout()
plt.savefig('profitability_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# YEARLY SALES & PROFIT TREND
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax2t = ax.twinx()

ax.bar(yearly['Order_Year'].astype(str), yearly['Total_Sales']/1000,
       color='#2196F3', alpha=0.7, label='Sales ($K)')
ax2t.plot(yearly['Order_Year'].astype(str), yearly['Total_Profit']/1000,
          'o-', color='#FF5722', linewidth=2.5, markersize=8, label='Profit ($K)')

ax.set_ylabel('Sales ($K)', color='#2196F3')
ax2t.set_ylabel('Profit ($K)', color='#FF5722')
ax.set_xlabel('Year')
ax.set_title('Annual Sales & Profit Trend', fontsize=14, fontweight='bold')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2t.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('yearly_trend.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. SQL-Based Aggregations

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOAD DATA INTO SQLITE IN-MEMORY DB
# ─────────────────────────────────────────────────────────────
conn = sqlite3.connect(':memory:')
df.to_sql('superstore', conn, index=False, if_exists='replace')
print("✅ Data loaded into SQLite in-memory database")

In [ ]:
# SQL Query 1 — Top 10 most profitable products
query1 = """
SELECT 
    "Product Name",
    Category,
    "Sub-Category",
    ROUND(SUM(Sales), 2)  AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct
FROM superstore
GROUP BY "Product Name", Category, "Sub-Category"
ORDER BY Total_Profit DESC
LIMIT 10;
"""
print("📊 Top 10 Most Profitable Products")
pd.read_sql(query1, conn)

In [ ]:
# SQL Query 2 — Loss-making states
query2 = """
SELECT 
    State,
    Region,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(SUM(Sales), 2)       AS Total_Sales,
    ROUND(SUM(Profit), 2)      AS Total_Profit
FROM superstore
GROUP BY State, Region
HAVING Total_Profit < 0
ORDER BY Total_Profit ASC;
"""
print("📊 Loss-Making States")
pd.read_sql(query2, conn)

In [ ]:
# SQL Query 3 — Shipping mode efficiency
query3 = """
SELECT 
    "Ship Mode",
    COUNT(*)                           AS Total_Orders,
    ROUND(AVG(Delivery_Time), 1)       AS Avg_Delivery_Days,
    ROUND(SUM(Sales), 2)               AS Total_Sales,
    ROUND(SUM(Profit), 2)              AS Total_Profit,
    ROUND(SUM(Profit)*100.0/SUM(Sales), 2) AS Profit_Margin_Pct
FROM superstore
GROUP BY "Ship Mode"
ORDER BY Avg_Delivery_Days;
"""
print("📊 Shipping Mode Efficiency")
pd.read_sql(query3, conn)

In [ ]:
# SQL Query 4 — Discount impact
query4 = """
SELECT 
    Discount_Bucket,
    COUNT(*)                        AS Orders,
    ROUND(AVG(Sales), 2)            AS Avg_Sales,
    ROUND(AVG(Profit), 2)           AS Avg_Profit,
    ROUND(AVG(Profit_Margin), 2)    AS Avg_Margin_Pct
FROM superstore
GROUP BY Discount_Bucket
ORDER BY Avg_Margin_Pct DESC;
"""
print("📊 Impact of Discounts on Profitability")
pd.read_sql(query4, conn)

## 9. Visualizations & Dashboards

In [ ]:
# ─────────────────────────────────────────────────────────────
# HEATMAP: PROFIT BY REGION & CATEGORY
# ─────────────────────────────────────────────────────────────
pivot = df.pivot_table(values='Profit', index='Region',
                       columns='Category', aggfunc='sum').round(0)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, center=0)
ax.set_title('Profit Heatmap: Region × Category ($)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('profit_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# MONTHLY SALES TREND (ALL YEARS)
# ─────────────────────────────────────────────────────────────
monthly = df.groupby(['Order_Year', 'Order_Month']).agg(
    Sales  = ('Sales', 'sum'),
    Profit = ('Profit', 'sum')
).reset_index()
monthly['Period'] = pd.to_datetime(monthly[['Order_Year', 'Order_Month']].rename(
    columns={'Order_Year': 'year', 'Order_Month': 'month'}).assign(day=1))

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly['Period'], monthly['Sales']/1000, alpha=0.3, color='#2196F3')
ax.plot(monthly['Period'], monthly['Sales']/1000, color='#2196F3', linewidth=1.5, label='Sales ($K)')
ax.plot(monthly['Period'], monthly['Profit']/1000, color='#FF5722', linewidth=1.5, label='Profit ($K)')
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.set_xlabel('Month')
ax.set_ylabel('Amount ($K)')
ax.set_title('Monthly Sales & Profit Trend (2014–2017)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('monthly_trend.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# TOP 10 STATES BY PROFIT
# ─────────────────────────────────────────────────────────────
state_profit = df.groupby('State')['Profit'].sum().sort_values()
top5_loss = state_profit.head(5)
top5_gain = state_profit.tail(5)
combined  = pd.concat([top5_loss, top5_gain])

fig, ax = plt.subplots(figsize=(10, 6))
colors_s = ['#FF5722' if v < 0 else '#4CAF50' for v in combined]
combined.plot(kind='barh', ax=ax, color=colors_s, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Total Profit ($)')
ax.set_title('Top 5 Profit & Loss States', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('state_profit.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Insights & Recommendations

In [ ]:
# ─────────────────────────────────────────────────────────────
# AUTOMATED INSIGHTS SUMMARY
# ─────────────────────────────────────────────────────────────
best_region   = region_stats.loc[region_stats['Profit'].idxmax(), 'Region']
worst_region  = region_stats.loc[region_stats['Profit'].idxmin(), 'Region']
best_cat      = df.groupby('Category')['Profit'].sum().idxmax()
worst_sub     = cat_stats.loc[cat_stats['Profit'].idxmin(), 'Sub-Category']
fastest_ship  = ship_stats.loc[ship_stats['Avg_Delivery'].idxmin(), 'Ship Mode']
std_ship_pct  = (ship_stats.loc[ship_stats['Ship Mode']=='Standard Class','Order_Count'].values[0]
                 / ship_stats['Order_Count'].sum() * 100)
loss_pct      = (~df['Is_Profitable']).mean() * 100
high_disc_loss= df[df['Discount'] > 0.4]['Profit'].mean()

print("=" * 60)
print("  📌  KEY INSIGHTS & RECOMMENDATIONS")
print("=" * 60)

insights = [
    ("💰 Best Region",      f"{best_region} generates the highest profit. Prioritize resource allocation here."),
    ("⚠️  Worst Region",    f"{worst_region} is underperforming. Investigate product mix and pricing."),
    ("📦 Best Category",    f"{best_cat} is the most profitable category. Consider expanding product range."),
    ("🔴 Loss Sub-Category",f"'{worst_sub}' is the biggest loss-maker. Review pricing or phase out unprofitable SKUs."),
    ("🚚 Shipping",         f"{fastest_ship} is fastest. {std_ship_pct:.1f}% of orders use Standard Class — consider incentivizing faster modes."),
    ("📉 Loss Orders",      f"{loss_pct:.1f}% of all orders are loss-making. Target discount capping."),
    ("🏷️  Discount Impact", f"Avg profit for >40% discount orders: ${high_disc_loss:.2f}. High discounts destroy margins."),
    ("📈 Growth Trend",     "Sales and profit both grew YoY. Maintain momentum while reducing cost leakages."),
]

for title, desc in insights:
    print(f"\n  {title}")
    print(f"    → {desc}")

print("\n" + "=" * 60)

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPORT CLEANED DATA
# ─────────────────────────────────────────────────────────────
df.to_csv('superstore_cleaned.csv', index=False)
print("✅ Cleaned dataset exported to 'superstore_cleaned.csv'")
print("✅ Analysis complete. All charts saved as PNG files.")